In [45]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline


# Data loading and cleaning

In [46]:
diamonds = pd.read_csv("../data/diamonds.csv")
# diamonds.info()
X = diamonds.drop("price", axis=1)
y = diamonds["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clarity_order = [
    "I3", "I2", "I1",
    "SI2", "SI1",
    "VS2", "VS1",
    "VVS2", "VVS1",
    "IF", "FL"
]

cut_order = [
    "Fair", "Good",
    "Very Good", "Premium",
    "Ideal"
]

color_order = [
    "J", "I", "H",
    "G", "F", "E", "D"
]

num_pipeline = make_pipeline(
    StandardScaler()
)

ordinal_pipeline = make_pipeline(
    OrdinalEncoder(categories=[clarity_order, cut_order, color_order]),
    StandardScaler()
)


preprocessing = ColumnTransformer([
    ("num", num_pipeline, list(X_train.select_dtypes(include=["int64", "float64"]).columns)),
    ("clarity", ordinal_pipeline, ["clarity", "cut", "color"])
])

# Model selection (Linear vs Ridge vs Lasso with cross_val_score)

In [47]:
alphas = np.logspace(-4, 4, 50)

cv_outer = KFold(n_splits=5, shuffle=True, random_state=42)

lin = make_pipeline(
    preprocessing,
    LinearRegression()
)

ridge = make_pipeline(
    preprocessing,
    RidgeCV( # auto cross validation for choosing alpha
        alphas=alphas,
        cv=5,
        scoring="r2"
    )
)

lasso = make_pipeline(
    preprocessing,
    LassoCV(  # auto cross validation for choosing alpha
        alphas=alphas,
        cv=5,
        max_iter=10000,
        random_state=42
    )
)

models = {
    "Linear Regression": lin,
    "RidgeCV": ridge,
    "LassoCV": lasso
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv_outer,
        scoring="r2"
    )
    
    print(f"{name}")
    print(f"R2 scores: {scores}")
    print(f"Mean R2: {scores.mean():.4f}")
    print(f"Std R2: {scores.std():.4f}")
    print()

Linear Regression
R2 scores: [0.9114471  0.91085104 0.89569444 0.91145881 0.90714021]
Mean R2: 0.9073
Std R2: 0.0060

RidgeCV
R2 scores: [0.91139808 0.91007115 0.89591624 0.91132204 0.90741284]
Mean R2: 0.9072
Std R2: 0.0058

LassoCV
R2 scores: [0.91140724 0.91066258 0.89587693 0.91133003 0.90756041]
Mean R2: 0.9074
Std R2: 0.0059



# Prediction

In [48]:
# we choose Lasso after our evaluations
final_model = lasso
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_test)
print("Test R2:", r2_score(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
print("Test MAE:", mean_absolute_error(y_test, y_pred))


Test R2: 0.9060946269620145
Test MSE: 1492796.9484987846
Test RMSE: 1221.8006991726534
Test MAE: 806.373822568731
